### Transformer-based Sentiment Analysis

In [1]:
#Run a pretrained transformer sentiment model on the SAME reviews at where TF-IDF model got wrong.
#Check whether attention/context fixes the "great, but..."
#Run the transformer on full test set and compare.

In [2]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix

pd.set_option("display.max_colwidth",150)

#Load pretrained model

print("Loading pretrained transformer(distilbert-base-uncased-finetuned-sst-2-english)")
#this model outputs only pos/neg -trained on movie reviews
#handle "neutral" via confidence threshold below
sentiment_model = pipeline("sentiment-analysis",model="distilbert-base-uncased-finetuned-sst-2-english")

def transformer_sentiment(text,neutral_threshold=0.65):
    result = sentiment_model(text[:512])[0]
    label, score = result["label"],result["score"]
    if score < neutral_threshold:
        return "neutral"
    return "positive" if label=="POSITIVE" else "negative"


Loading pretrained transformer(distilbert-base-uncased-finetuned-sst-2-english)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\Priyanka\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Priyanka\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [10]:
#Run on misclassified by tfidf

print("\n Loading misclassified examples...")
misclassified = pd.read_csv("C:/Users/Priyanka/Datascience/NLP_projects/missclassified.csv")

print(f"Running transformer on{len(misclassified)} examples TF-IDF got wrong\n")
misclassified["transformer_pred"]=misclassified["text"].apply(transformer_sentiment)

#did the transformer fix what tf-idf got wrong?
misclassified["transformer_correct"]=(misclassified["transformer_pred"] == misclassified["actual"])

print("\nLabel distribution:")

print("\nActual:")
print(misclassified["actual"].value_counts())

print("\nTransformer:")
print(misclassified["transformer_pred"].value_counts())

fix_rate = misclassified["transformer_correct"].mean() # improvement than tf-idf
print(f"Transformer got {fix_rate:.1%} of TF-IDF's mistakes right.\n")

print("\nSample Conversion(but,great contrastive reviews)")
cols=["text","actual","predicted","transformer_pred"]
print(misclassified.rename(columns={"predicted":"tfidf_pred"})[["text","actual","transformer_pred"]].head(10))

misclassified.to_csv("transformer_vs_tfidf.csv", index=False)
print("\n saved successfully")


 Loading misclassified examples...
Running transformer on1763 examples TF-IDF got wrong


Label distribution:

Actual:
actual
positive    1244
negative     292
neutral      227
Name: count, dtype: int64

Transformer:
transformer_pred
negative    941
positive    759
neutral      63
Name: count, dtype: int64
Transformer got 53.2% of TF-IDF's mistakes right.


Sample Conversion(but,great contrastive reviews)
                                                                                                                                                    text  \
0                                this has got to be the best black licorice in the world as i consider myself quite the connoisseur of this fine product   
1  this is what you have been looking for unless you grind your own annato seeds if you re wanting that yellow rice you ve had in miami puerto rico b...   
2  i took him off the pedigree food after i heard about how loq quality it is purina one isn t the highest grade dog food 

In [11]:
# full head-to-head on test_sample

df = pd.read_csv("C:/Users/Priyanka/Datascience/NLP_projects/cleaned_data.csv")

sample = df.sample(n=1000, random_state=42).reset_index(drop=True)
sample["transformer_pred"] = sample["clean_text"].apply( transformer_sentiment)
print("\n====================================================================")
print("TRANSFORMER CLASSIFICATION REPORT(1000 SAMPLE)/n")
print("\n====================================================================")
print(classification_report(sample["sentiment"], sample["transformer_pred"]))
cm = confusion_matrix(
    sample["sentiment"], sample["transformer_pred"],
    labels=["negative", "neutral", "positive"]
)
print("Confusion Matrix (rows=actual, cols=predicted):")
print(pd.DataFrame(cm, index=["neg", "neu", "pos"], columns=["neg", "neu", "pos"]))


TRANSFORMER CLASSIFICATION REPORT(1000 SAMPLE)/n

              precision    recall  f1-score   support

    negative       0.39      0.91      0.55       169
     neutral       0.19      0.07      0.11        67
    positive       0.96      0.73      0.83       764

    accuracy                           0.72      1000
   macro avg       0.51      0.57      0.49      1000
weighted avg       0.81      0.72      0.73      1000

Confusion Matrix (rows=actual, cols=predicted):
     neg  neu  pos
neg  154    0   15
neu   51    5   11
pos  185   22  557


In [12]:
sample.to_csv("transformer_full_sample.csv",index=False)
